In [1]:
# %pip install polars
# %pip install duckdb

In [1]:
import pandas as pd
import duckdb
import polars as pl

In [4]:
# Load parquet into lazy dataframe and display the head
PARQUET_PATH = '../data/processed/consumer_banking_complaints.parquet'
df = pl.scan_parquet(PARQUET_PATH)

df.head(5).collect()

Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
date,str,str,str,str,str,str,str,str,str,str,str,str,date,str,bool,str,i64
2019-12-26,"""Credit card or prepaid card""","""General-purpose credit card or…","""Advertising and marketing, inc…","""Confusing or misleading advert…",null,null,"""CAPITAL ONE FINANCIAL CORPORAT…","""CA""","""94025""",null,"""Consent not provided""","""Web""",2019-12-26,"""Closed with explanation""",true,"""N/A""",3477549
2019-12-20,"""Checking or savings account""","""Other banking product or servi…","""Managing an account""","""Funds not handled or disbursed…",null,"""Company has responded to the c…","""WELLS FARGO & COMPANY""","""FL""","""33064""",null,"""N/A""","""Referral""",2019-12-23,"""Closed with explanation""",true,"""N/A""",3475858
2019-11-18,"""Credit card or prepaid card""","""General-purpose credit card or…","""Problem with a purchase shown …","""Credit card company isn't reso…","""XXXX claimed they delivered a …",null,"""DISCOVER BANK""","""MA""","""021XX""",null,"""Consent provided""","""Web""",2019-11-18,"""Closed with explanation""",true,"""N/A""",3442136
2020-06-05,"""Checking or savings account""","""Checking account""","""Managing an account""","""Problem using a debit or ATM c…",null,"""Company has responded to the c…","""CITIBANK, N.A.""","""NY""","""10466""",null,"""Consent not provided""","""Web""",2020-06-05,"""Closed with explanation""",true,"""N/A""",3684669
2024-01-16,"""Credit card""","""General-purpose credit card or…","""Other features, terms, or prob…","""Add-on products and services""",null,"""Company has responded to the c…","""WELLS FARGO & COMPANY""","""TX""","""76179""",null,"""Consent not provided""","""Web""",2024-01-16,"""Closed with monetary relief""",true,"""N/A""",8161600


In [7]:
# Complaints by product
duckdb.sql(f"""
    SELECT
        Product,
        COUNT(*) AS complaints
    FROM '{PARQUET_PATH}'
    GROUP BY Product
    ORDER BY complaints DESC
""")

┌─────────────────────────────┬────────────┐
│           Product           │ complaints │
│           varchar           │   int64    │
├─────────────────────────────┼────────────┤
│ Checking or savings account │     368590 │
│ Mortgage                    │     275786 │
│ Credit card                 │     254494 │
│ Credit card or prepaid card │     206364 │
│ Bank account or service     │      28801 │
└─────────────────────────────┴────────────┘

In [8]:
# Number of records with non-null narratives - all products
count_narratives_all = (
    df
    .filter(
        pl.col('Consumer complaint narrative').is_not_null()
    )
    .select(pl.len().alias('count_narratives_all'))
    .collect()
)

print(f'The number of records with valid narratives: {count_narratives_all}')

The number of records with valid narratives: shape: (1, 1)
┌──────────────────────┐
│ count_narratives_all │
│ ---                  │
│ u32                  │
╞══════════════════════╡
│ 541497               │
└──────────────────────┘


In [10]:
# Top 10 companies - valid narratives

duckdb.sql(f"""
    SELECT
        Company,
        COUNT(*) AS narrative_count
    FROM '{PARQUET_PATH}'
    WHERE "Consumer complaint narrative" IS NOT NULL
    GROUP BY Company
    ORDER BY narrative_count DESC
    LIMIT 10
""")

┌───────────────────────────────────────┬─────────────────┐
│                Company                │ narrative_count │
│                varchar                │      int64      │
├───────────────────────────────────────┼─────────────────┤
│ JPMORGAN CHASE & CO.                  │           44623 │
│ WELLS FARGO & COMPANY                 │           40269 │
│ BANK OF AMERICA, NATIONAL ASSOCIATION │           39751 │
│ CITIBANK, N.A.                        │           38842 │
│ CAPITAL ONE FINANCIAL CORPORATION     │           38299 │
│ SYNCHRONY FINANCIAL                   │           19789 │
│ AMERICAN EXPRESS COMPANY              │           15624 │
│ NAVY FEDERAL CREDIT UNION             │           14912 │
│ U.S. BANCORP                          │           13194 │
│ Chime Financial Inc                   │           12006 │
└───────────────────────────────────────┴─────────────────┘
  10 rows                                       2 columns

In [12]:
# # Print sample of 10 narratives
# sample_narratives = (
#     df
#     .filter(pl.col('Consumer complaint narrative').is_not_null())
#     .select('Consumer complaint narrative')
#     .collect()
#     .sample(n=10, shuffle=True)
# )
#
# narratives = sample_narratives['Consumer complaint narrative'].to_list()
#
# for i, narrative in enumerate(narratives, start=1):
#     print(f'\n--- Narrative {i} ---')
#     print(narrative)

In [17]:
# Load parquet into pandas dataframe
pandas_df = pd.read_parquet(f'{PARQUET_PATH}')

In [18]:
pandas_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1134035 entries, 0 to 1134034
Data columns (total 18 columns):
 #   Column                        Non-Null Count    Dtype 
---  ------                        --------------    ----- 
 0   Date received                 1134035 non-null  object
 1   Product                       1134035 non-null  str   
 2   Sub-product                   1105736 non-null  str   
 3   Issue                         1134030 non-null  str   
 4   Sub-issue                     869805 non-null   str   
 5   Consumer complaint narrative  541497 non-null   str   
 6   Company public response       562889 non-null   str   
 7   Company                       1134035 non-null  str   
 8   State                         1114965 non-null  str   
 9   ZIP code                      1120163 non-null  str   
 10  Tags                          201643 non-null   str   
 11  Consumer consent provided?    1084329 non-null  str   
 12  Submitted via                 1134035 non-null  str  

In [20]:
# Value counts in "Timely response?"
pandas_df['Timely response?'].value_counts()

Timely response?
True     1120957
False      13078
Name: count, dtype: int64

In [21]:
# Percentage of records with non-blank/non-null narratives
count_non_null = pandas_df['Consumer complaint narrative'].count()
print(f'The number of records in the df (5 products) with non-blank/non-null narratives: {count_non_null}')
total_rows = len(pandas_df)

pct_with_narratives = round(((count_non_null / total_rows) * 100), 2)
print(f'The percentage of records with non-blank complaint narratives is: {pct_with_narratives:.2f}%')

The number of records in the df (5 products) with non-blank/non-null narratives: 541497
The percentage of records with non-blank complaint narratives is: 47.75%


In [22]:
# Shift back to parquet file for SQL queries
# Count narratives by year - all products
duckdb.sql(f"""
    SELECT
        YEAR("Date received") AS year,
        COUNT(*) AS complaints,
        COUNT("Consumer complaint narrative") AS narratives
    FROM '{PARQUET_PATH}'
    GROUP BY year
    ORDER BY year
""")

┌───────┬────────────┬────────────┐
│ year  │ complaints │ narratives │
│ int64 │   int64    │   int64    │
├───────┼────────────┼────────────┤
│  2016 │      84365 │      32953 │
│  2017 │      72811 │      31399 │
│  2018 │      70017 │      29123 │
│  2019 │      70264 │      29913 │
│  2020 │      82769 │      39099 │
│  2021 │      87911 │      43087 │
│  2022 │     100751 │      51812 │
│  2023 │     129910 │      75089 │
│  2024 │     150258 │      77147 │
│  2025 │     198878 │     106940 │
│  2026 │      86101 │      24935 │
└───────┴────────────┴────────────┘
  11 rows               3 columns

In [27]:
# Narratives by sub-product
duckdb.sql(f"""
    SELECT
        Product,
        "Sub-product",
        COUNT(*) AS complaints,
        COUNT("Consumer complaint narrative") AS narratives
    FROM '{PARQUET_PATH}'
    GROUP BY Product, "Sub-product"
    ORDER BY Product, narratives DESC
""")

┌─────────────────────────────┬────────────────────────────────────────────┬────────────┬────────────┐
│           Product           │                Sub-product                 │ complaints │ narratives │
│           varchar           │                  varchar                   │   int64    │   int64    │
├─────────────────────────────┼────────────────────────────────────────────┼────────────┼────────────┤
│ Bank account or service     │ Checking account                           │      17777 │       7057 │
│ Bank account or service     │ Other bank product/service                 │       8425 │       2390 │
│ Bank account or service     │ Savings account                            │       1637 │        614 │
│ Bank account or service     │ (CD) Certificate of deposit                │        770 │        176 │
│ Bank account or service     │ Cashing a check without an account         │        192 │         87 │
│ Checking or savings account │ Checking account                         

In [23]:
# Min/Max and Avg characters in the narratives
duckdb.sql(f"""
    SELECT
        MIN(LENGTH("Consumer complaint narrative")) AS min_chars,
        AVG(LENGTH("Consumer complaint narrative")) AS avg_chars,
        MAX(LENGTH("Consumer complaint narrative")) AS max_chars
    FROM '{PARQUET_PATH}'
    WHERE "Consumer complaint narrative" IS NOT NULL;
""")

┌───────────┬────────────────────┬───────────┐
│ min_chars │     avg_chars      │ max_chars │
│   int64   │       double       │   int64   │
├───────────┼────────────────────┼───────────┤
│        13 │ 1666.9142066225725 │     32317 │
└───────────┴────────────────────┴───────────┘

In [28]:
# Top 20 longest narratives with product and sub-product
duckdb.sql(f"""
    SELECT
        LENGTH("Consumer complaint narrative") AS chars,
        "Issue",
        "Sub-product"
    FROM '{PARQUET_PATH}'
    WHERE "Consumer complaint narrative" IS NOT NULL
    ORDER BY chars DESC
    LIMIT 20;
""")

┌───────┬─────────────────────────────────────────────────────────────────┬────────────────────────────────────────────┐
│ chars │                              Issue                              │                Sub-product                 │
│ int64 │                             varchar                             │                  varchar                   │
├───────┼─────────────────────────────────────────────────────────────────┼────────────────────────────────────────────┤
│ 32785 │ Advertising and marketing, including promotional offers         │ General-purpose credit card or charge card │
│ 32763 │ Closing your account                                            │ General-purpose credit card or charge card │
│ 32555 │ Closing your account                                            │ General-purpose credit card or charge card │
│ 32549 │ Closing your account                                            │ General-purpose credit card or charge card │
│ 32545 │ Closing your account  

In [29]:
# Narratives by issue
duckdb.sql(f"""
SELECT
    Issue,
    COUNT(*) AS narratives
FROM '{PARQUET_PATH}'
WHERE "Consumer complaint narrative" IS NOT NULL
GROUP BY Issue
ORDER BY narratives DESC;
""")

┌──────────────────────────────────────────────────────────────┬────────────┐
│                            Issue                             │ narratives │
│                           varchar                            │   int64    │
├──────────────────────────────────────────────────────────────┼────────────┤
│ Managing an account                                          │      97151 │
│ Trouble during payment process                               │      54832 │
│ Problem with a purchase shown on your statement              │      51762 │
│ Struggling to pay mortgage                                   │      26316 │
│ Closing an account                                           │      24742 │
│ Other features, terms, or problems                           │      24266 │
│ Fees or interest                                             │      22354 │
│ Problem with a lender or other company charging your account │      21915 │
│ Getting a credit card                                        │

In [30]:
# Avg narrative length by issue
duckdb.sql(f"""
    SELECT
        Issue,
        AVG(LENGTH("Consumer complaint narrative")) AS avg_chars
    FROM '{PARQUET_PATH}'
    WHERE "Consumer complaint narrative" IS NOT NULL
    GROUP BY Issue
    ORDER BY avg_chars DESC;
""")

┌──────────────────────────────────────────────────────────────────────────────────┬────────────────────┐
│                                      Issue                                       │     avg_chars      │
│                                     varchar                                      │       double       │
├──────────────────────────────────────────────────────────────────────────────────┼────────────────────┤
│ Closing on a mortgage                                                            │ 1941.1920460986207 │
│ Struggling to pay mortgage                                                       │ 1925.5790773673812 │
│ Applying for a mortgage or refinancing an existing mortgage                      │ 1750.9236698499317 │
│ Trouble during payment process                                                   │ 1640.6124161073826 │
│ Loan modification,collection,foreclosure                                         │ 1586.9921089277425 │
│ Problem with a purchase shown on your statem

In [31]:
# narrative length vs. timely response
duckdb.sql(f"""
    SELECT
        "Timely response?",
        AVG(LENGTH("Consumer complaint narrative")) AS avg_chars
    FROM '{PARQUET_PATH}'
    WHERE "Consumer complaint narrative" IS NOT NULL
    GROUP BY "Timely response?";
""")

┌──────────────────┬────────────────────┐
│ Timely response? │     avg_chars      │
│     boolean      │       double       │
├──────────────────┼────────────────────┤
│ true             │ 1328.4390426187852 │
│ false            │ 1458.2104342293621 │
└──────────────────┴────────────────────┘